## Bài tập thực hành 2 - 2026
## Phân loại ảnh sử dụng Phương pháp phân loại tuyến tính với các đặc trưng màu sắc
Link google colab: https://drive.google.com/file/d/1Qrp6bia-DhiF6NhJyfshRi2mwnwkl3g3/view?usp=sharing

---
### Thông tin sinh viên
- **MSSV**: 24521901
- **Họ và tên**: Trần Quang Trường
---


### 0. Mô tả bài toán

Cho trước 1 tập dữ liệu HoaVietNam2026.

Gọi X là đặc trưng có Độ chính xác (Accuracy) trong bài tập 1 trên tập dữ liệu HoaVietnam2026.

---

#### 0.1 Đọc dữ liệu
Trước khi thực hiện yêu cầu bài tập, ta sẽ đọc tập dữ liệu train và test từ `HoaVietNam2026.zip`

Để truy cập tập dữ liệu, ta sẽ mount tới Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import các thư viện cần thiết


In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

Vì cấu trúc tập train và test là như nhau nên ta sẽ tạo một hàm để có thể extract dữ liệu từ thư mục sau khi giải nén.

In [3]:
def load_images_from_folder(folder_dir):
    images = []
    labels = []
    class_names = sorted(os.listdir(folder_dir))

    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder_dir, class_name)
        if os.path.isdir(class_folder):
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)

    return images, labels, class_names

In [4]:
train_folder_dir  = '/content/drive/MyDrive/[CS231.Q21.KHTN] - Computer Vision 01/HoaVietNam2026/train'
test_folder_dir   = '/content/drive/MyDrive/[CS231.Q21.KHTN] - Computer Vision 01/HoaVietNam2026/test'

X_train, y_train, class_names = load_images_from_folder(train_folder_dir)
X_test, y_test, _ = load_images_from_folder(test_folder_dir)

y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Nhãn các loài hoa: {class_names}")
print(f"Tổng số ảnh trong tập train: {len(X_train)}")
print(f"Tổng số ảnh trong tập test: {len(X_test)}")

Nhãn các loài hoa: ['Cuc', 'Dao', 'Ga', 'Lan', 'Mai', 'Sen', 'Tho']
Tổng số ảnh trong tập train: 250
Tổng số ảnh trong tập test: 77


Ở đây, ta sẽ sử dụng đặc trưng màu sắc **Historgram với 3 thành phần màu HSV tương ứng 8x8x8** với acc cao nhất trong các đặc trưng là 0.7403 như đã trình bày ở bài tập trước.

In [5]:
#Histogram với 3 thành phần màu HSV tương ứng 8x8x8
def compute_histogram_feature_hsv(image, bins=8):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([image], [0, 1, 2], None, [bins, bins, bins], [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return np.array(hist)

In [6]:
X_train_hsv8 = np.array([compute_histogram_feature_hsv(img) for img in X_train])
X_test_hsv8 = np.array([compute_histogram_feature_hsv(img) for img in X_test])

---

### 1. Yêu cầu 1
Sử dụng bộ Phân lớp Linear Classifer: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

Khởi tạo mô hình LogisticRegression:

In [7]:
logistic = LogisticRegression(max_iter=100)

Định nghĩa các tham số cho GridSearchCV:

In [8]:
param_grid_logistic = {
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
}


Huấn luận mô hình để tìm kiếm bộ tham số tốt nhất:

In [9]:
grid_search_logistic = GridSearchCV(logistic, param_grid_logistic, scoring='f1_macro', cv=5, n_jobs=-1)
grid_search_logistic.fit(X_train_hsv8, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
55 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

GridSearchCV(cv=5, estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['lbfgs', 'liblinear', 'newton-cg',
                                    'newton-cholesky', 'sag', 'saga']},
             scoring='f1_macro')

Đưa ra tham số tương ứng kết quả độ đô Macro F1-score tốt nhất tìm được:

In [10]:
print("Best Parameters:",  grid_search_logistic.best_params_)
print("Best Macro F1-Score:",grid_search_logistic.best_score_)

Best Parameters: {'penalty': None, 'solver': 'sag'}
Best Macro F1-Score: 0.763450312035009


---

### 2. Yêu cầu 2
Dùng hàm classification_report để in kết quả phân loại

In [11]:
best_model_logistic = grid_search_logistic.best_estimator_
y_pred_hsv8_logistic = best_model_logistic.predict(X_test_hsv8)

test_f1_macro = f1_score(y_test, y_pred_hsv8_logistic, average='macro')

print(f"Test Macro F1-Score: {test_f1_macro:.4f}")

print(f"Classification Report:")
print(classification_report(y_test, y_pred_hsv8_logistic))

Test Macro F1-Score: 0.6937
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.70      0.74        10
           1       0.69      0.75      0.72        12
           2       0.79      0.73      0.76        15
           3       0.56      0.50      0.53        10
           4       0.70      0.70      0.70        10
           5       0.67      0.60      0.63        10
           6       0.69      0.90      0.78        10

    accuracy                           0.70        77
   macro avg       0.70      0.70      0.69        77
weighted avg       0.70      0.70      0.70        77



Thực hiện tương tự như yêu cầu 1 nhưng sử dụng phương pháp [SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html)

Khởi tạo mô hình SGDClassifer:

In [12]:

sgd = SGDClassifier(loss='log_loss', max_iter=100)

Khởi tạo các tham số cho GridSearchCV:

In [13]:
param_grid_sgd = {
    'penalty': ['l2', 'l1', 'elasticnet', None],
    'learning_rate': ['optimal', 'adaptive']
}

Huấn luận mô hình để tìm kiếm bộ tham số tốt nhất:

In [14]:
grid_search_sgd = GridSearchCV(sgd, param_grid_sgd, scoring='f1_macro', cv=5, n_jobs=-1)
grid_search_sgd.fit(X_train_hsv8, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py", line 930, in fit
    self._more_v

GridSearchCV(cv=5, estimator=SGDClassifier(loss='log_loss', max_iter=100),
             n_jobs=-1,
             param_grid={'learning_rate': ['optimal', 'adaptive'],
                         'penalty': ['l2', 'l1', 'elasticnet', None]},
             scoring='f1_macro')

Đưa ra tham số tương ứng kết quả độ đô Macro F1-score tốt nhất tìm được:

In [15]:
print("Best Parameters:",  grid_search_sgd.best_params_)
print("Best Macro F1-Score:",grid_search_sgd.best_score_)

Best Parameters: {'learning_rate': 'optimal', 'penalty': 'l2'}
Best Macro F1-Score: 0.7425330738355947


In [16]:
best_model_sgd = grid_search_sgd.best_estimator_
y_pred_hsv8_sgd = best_model_sgd.predict(X_test_hsv8)

test_f1_macro = f1_score(y_test, y_pred_hsv8_sgd, average='macro')

print(f"Test Macro F1-Score: {test_f1_macro:.4f}")

print(f"Classification Report:")
print(classification_report(y_test, y_pred_hsv8_sgd))

Test Macro F1-Score: 0.7380
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.70      0.70        10
           1       0.83      0.83      0.83        12
           2       0.77      0.67      0.71        15
           3       0.60      0.60      0.60        10
           4       0.75      0.90      0.82        10
           5       0.70      0.70      0.70        10
           6       0.80      0.80      0.80        10

    accuracy                           0.74        77
   macro avg       0.74      0.74      0.74        77
weighted avg       0.74      0.74      0.74        77

